## Dados do FUNDEB por município

**Origem:** `fundeb/fundeb-por-municipio.csv` (separador `;`, encoding `latin-1`).

**Estrutura do arquivo:**
- Cada município aparece em **12 linhas**, uma por mês (`Mês` de 1 a 12). São 5.570 municípios × 12 meses = 66.840 linhas, mais 3 linhas vazias no fim do arquivo, removidas na leitura.
- As colunas `COD_MUN`, `Município`, `UF` e `Município - UF` identificam o município.
- As colunas `2007` a `2026` são os anos. Cada valor é o montante do FUNDEB daquele município naquele **mês** e **ano**, ou seja, a interseção da linha (município + mês) com a coluna do ano.
- Os valores vêm como texto no formato brasileiro, com ponto de milhar e vírgula decimal (ex.: `1.926.974,37`), e às vezes com espaços. Por isso são convertidos para `float` na célula de agrupamento.
- As colunas `Unnamed: 25` e `Unnamed: 26` estão vazias e são resultado do `;` no fim das linhas.

**Código do município:** a coluna `COD_MUN` **não** corresponde ao código IBGE de 7 dígitos. Por exemplo, Acrelândia é `0643` no fundeb e `1200013` no IBGE. Para relacionar com outros datasets, como o `all_students`, que usa `CO_MUNICIPIO`, é preciso cruzar por **sigla da UF + nome do município**, normalizado (sem acento e em maiúsculas). A comparação está no fim deste notebook.

**Agrupamento por ano:** para facilitar o relacionamento, os 12 meses são somados por município e ano, gerando `fundeb_anual` com uma linha por ano e município (colunas `year`, `city_code`, `city_name`, `state_abbreviation` e `fundeb_amount`). Gerado por Claude.

In [1]:
import pandas as pd
import gc

In [2]:
fundeb = pd.read_csv("./fundeb/fundeb-por-municipio.csv", sep=";", encoding="latin-1")
fundeb.sort_values(by=['COD_MUN', 'Município', 'Mês']).head(24)

,COD_MUN,Município,UF,Município - UF,Mês,2007,2008,2009,2010,2011,...,2019,2020,2021,2022,2023,2024,2025,2026,Unnamed: 25,Unnamed: 26
46464,1.0,Guajará-Mirim,RO,Guajará-Mirim - RO,1.0,"323.954,14","626.522,99","654.401,93","664.354,10","972.643,23",...,"1.612.894,04","1.439.647,74","2.175.744,29","1.951.062,93","1.558.316,34","2.203.953,48","2.678.670,55","3.004.479,20",NaN,NaN
46465,1.0,Guajará-Mirim,RO,Guajará-Mirim - RO,2.0,"283.197,61","596.384,99","571.629,38","674.473,76","925.424,77",...,"1.832.889,53","1.675.890,36","1.914.237,97","2.290.505,30","2.456.045,77","2.827.364,58","2.748.228,67","3.037.351,58",NaN,NaN
46466,1.0,Guajará-Mirim,RO,Guajará-Mirim - RO,3.0,"346.342,38","395.464,28","527.486,63","559.551,45","770.789,18",...,"1.412.916,54","1.725.793,09","1.727.205,22","1.858.621,87","1.880.997,41","1.896.650,45","2.510.204,24","2.464.504,34",NaN,NaN
46467,1.0,Guajará-Mirim,RO,Guajará-Mirim - RO,4.0,"458.472,56","443.211,68","646.104,58","685.901,43","851.785,98",...,"1.502.757,72","1.309.901,90","1.809.118,91","1.954.904,39","2.003.070,38","1.987.753,15","2.406.378,47","3.266.076,80",NaN,NaN
46468,1.0,Guajará-Mirim,RO,Guajará-Mirim - RO,5.0,"628.563,28","477.795,78","580.905,36","759.129,12","956.571,79",...,"1.514.796,21","1.345.967,50","1.465.438,96","2.139.867,25","2.451.850,42","2.704.933,46","2.985.851,88","3.214.383,99",NaN,NaN
46469,1.0,Guajará-Mirim,RO,Guajará-Mirim - RO,6.0,"428.915,07","447.981,74","517.101,22","735.993,12","974.060,88",...,"1.782.552,15","1.338.743,27","1.925.715,90","2.113.965,40","2.088.263,36","2.227.372,74","2.781.859,99","3.099.461,66",NaN,NaN
46470,1.0,Guajará-Mirim,RO,Guajará-Mirim - RO,7.0,"382.049,91","468.738,96","461.456,27","697.582,28","891.326,69",...,"1.571.765,05","1.407.880,85","1.947.508,60","2.090.878,54","1.974.311,84","2.257.794,11","2.745.843,85","3.075.214,32",NaN,NaN
46471,1.0,Guajará-Mirim,RO,Guajará-Mirim - RO,8.0,"408.576,63","547.028,03","627.818,36","726.232,49","904.604,69",...,"1.611.148,31","1.588.653,73","2.111.528,16","1.991.803,49","2.004.451,95","2.813.280,36","2.642.035,03","3.938.911,26",NaN,NaN
46472,1.0,Guajará-Mirim,RO,Guajará-Mirim - RO,9.0,"404.663,45","634.537,47","613.881,05","718.415,14","854.720,84",...,"1.640.130,94","1.457.518,66","1.851.658,63","1.758.787,07","1.877.949,40","1.667.701,61","2.301.982,31","2.527.872,57",NaN,NaN
46473,1.0,Guajará-Mirim,RO,Guajará-Mirim - RO,10.0,"386.791,96","601.884,87","604.995,69","758.886,77","880.070,18",...,"1.457.793,46","1.600.256,99","1.863.227,52","1.773.879,27","1.651.635,20","2.470.455,14","2.913.216,74",-,NaN,NaN


In [4]:
# Renomear colunas
fundeb = fundeb.rename(columns={
    'Município': 'city_name',
    'UF': 'state_abbreviation',
    'Mês': 'month'})

# Remove as 3 linhas vazias
fundeb = fundeb.dropna(subset=['city_name'])

# Mudar city_code e month para int
fundeb['month'] = fundeb['month'].astype(int)

fundeb.sort_values(by=['city_name', 'month']).head(24)

,COD_MUN,city_name,state_abbreviation,Município - UF,month,2007,2008,2009,2010,2011,...,2019,2020,2021,2022,2023,2024,2025,2026,Unnamed: 25,Unnamed: 26
10584,1050.0,Abadia de Goiás,GO,Abadia de Goiás - GO,1,"118.394,27","126.706,96","161.018,02","142.462,08","210.055,44",...,"760.564,39","860.770,34","1.049.861,53","1.409.745,19","1.383.650,06","2.054.805,06","3.337.909,63","4.763.955,58",NaN,NaN
10585,1050.0,Abadia de Goiás,GO,Abadia de Goiás - GO,2,"112.614,59","130.161,70","140.213,43","170.095,22","206.455,39",...,"767.691,42","883.560,65","1.076.702,87","1.435.163,46","1.522.412,69","2.286.558,58","2.637.508,57","2.963.432,61",NaN,NaN
10586,1050.0,Abadia de Goiás,GO,Abadia de Goiás - GO,3,"85.484,64","130.619,49","152.979,00","195.212,64","247.265,56",...,"708.913,53","799.371,59","1.008.767,31","1.366.131,44","1.356.642,28","1.897.618,64","2.594.024,87","2.676.461,01",NaN,NaN
10587,1050.0,Abadia de Goiás,GO,Abadia de Goiás - GO,4,"76.398,56","138.219,11","136.333,35","169.217,88","233.868,79",...,"719.343,33","768.139,37","1.154.100,22","1.429.836,71","1.394.438,28","2.225.269,15","2.173.070,77","3.130.509,24",NaN,NaN
10588,1050.0,Abadia de Goiás,GO,Abadia de Goiás - GO,5,"91.317,68","153.481,66","156.341,50","218.152,67","240.787,95",...,"859.162,64","647.232,10","1.156.168,47","1.451.831,62","1.471.588,74","2.214.757,07","3.348.773,21","3.086.377,62",NaN,NaN
10589,1050.0,Abadia de Goiás,GO,Abadia de Goiás - GO,6,"110.670,83","150.725,89","151.702,34","216.143,76","240.538,22",...,"712.630,14","791.099,99","1.102.308,45","1.535.408,80","1.514.178,32","2.327.390,93","2.836.104,14","3.051.778,80",NaN,NaN
10590,1050.0,Abadia de Goiás,GO,Abadia de Goiás - GO,7,"99.698,85","153.672,34","158.094,04","170.630,30","222.628,58",...,"736.051,05","887.367,48","1.312.407,95","1.370.503,20","1.339.010,34","2.138.060,42","2.704.194,97","3.659.542,12",NaN,NaN
10591,1050.0,Abadia de Goiás,GO,Abadia de Goiás - GO,8,"98.160,08","154.388,95","154.974,27","199.110,76","229.346,38",...,"820.209,80","963.274,72","1.399.844,59","1.300.777,68","1.557.443,53","2.506.531,81","2.804.321,18","3.389.047,48",NaN,NaN
10592,1050.0,Abadia de Goiás,GO,Abadia de Goiás - GO,9,"119.624,16","148.300,42","144.115,04","197.820,81","257.450,66",...,"754.030,81","858.897,73","1.387.159,65","1.432.894,34","1.542.146,38","2.286.636,84","2.262.636,77","3.341.308,78",NaN,NaN
10593,1050.0,Abadia de Goiás,GO,Abadia de Goiás - GO,10,"101.158,25","155.232,79","173.222,55","183.937,74","217.318,12",...,"886.952,22","1.609.240,17","1.442.489,13","1.490.111,17","1.882.614,36","2.948.630,82","3.785.656,08",-,NaN,NaN


In [55]:
# Lista dos anos de 2023 a 2025
anos = [str(ano) for ano in range(2023, 2026)]

# Converte o texto no formato brasileiro ("1.926.974,37") para float; "-" vira NaN
fundeb_anos = fundeb[anos].apply(lambda col: pd.to_numeric(col.str.replace('.', '', regex=False).str.replace(',', '.', regex=False), errors='coerce'))

# Junta as colunas de identificação com os valores anuais
fundeb_anual = pd.concat([fundeb[['city_name', 'state_abbreviation']], fundeb_anos], axis=1)

# Agrupa por município e soma os valores dos meses de cada ano
fundeb_anual = fundeb_anual.groupby(['city_name', 'state_abbreviation'], as_index=False)[anos].sum()

# Passa os anos de colunas para linhas: uma linha por município e ano
fundeb_anual = fundeb_anual.melt(id_vars=['city_name', 'state_abbreviation'], value_vars=anos, var_name='year', value_name='fundeb_amount')

# Converte o ano para inteiro e ordena por ano e nome do município
fundeb_anual['year'] = fundeb_anual['year'].astype(int)
# Coloca year como primeira coluna, antes de city_code
fundeb_anual = fundeb_anual[['year', 'city_name', 'state_abbreviation', 'fundeb_amount']]

fundeb_anual = fundeb_anual.sort_values(['year', 'city_name']).reset_index(drop=True)

fundeb_anual.to_csv("fundeb/fundeb.csv", sep=";", index=False, encoding="utf-8")
fundeb_anual.head(10)

,year,city_code,city_name,state_abbreviation,fundeb_amount
0,2023,1,Guajará-Mirim,RO,2.503422e+07
1,2023,2,Alto Alegre dos Parecis,RO,1.516151e+07
2,2023,3,Porto Velho,RO,3.238528e+08
3,2023,4,Buritis,RO,2.095758e+07
4,2023,5,Ji-Paraná,RO,6.029253e+07
5,2023,6,Chupinguaia,RO,1.085851e+07
6,2023,7,Ariquemes,RO,8.240671e+07
7,2023,8,Cujubim,RO,2.020493e+07
8,2023,9,Cacoal,RO,5.062223e+07
9,2023,10,Nova União,RO,6.521052e+06
